# 00 — Dataset Creation

Takes raw Nepali financial document images from `data/raw/` and produces a
**class-balanced** orientation dataset.

## Class balance strategy

All source PDFs are naturally 0° — naive collection gives 100% class 0.
We enforce balance by construction:

1. **Rotate** every source image × 4 orientations (0°, 90°, 180°, 270°)
   → guaranteed 25% per class
2. **Augment** every rotated image with scanner-realistic noise
   → model can't use image quality as an orientation cue

Output: `data/dataset.csv` with columns: `image_path, label, degrees, source, augmented`

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from src.preprocess import rotate_image, DEGREES_TO_LABEL

RAW_DIR  = Path('../data/raw')
OUT_DIR  = Path('../data/rotated')
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tiff', '.bmp'}

In [ ]:
# ── Augmentation helpers ──────────────────────────────────────────────────────
# Applied to ALL orientations equally so image quality is not an orientation cue.

rng = np.random.default_rng(seed=42)

def augment(img: np.ndarray) -> np.ndarray:
    """Apply scanner-realistic augmentation to an HWC uint8 BGR image."""
    img = img.copy()

    # 1. Brightness / contrast jitter  (simulate scanner exposure variation)
    alpha = rng.uniform(0.85, 1.15)   # contrast
    beta  = rng.integers(-15, 15)     # brightness
    img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)

    # 2. Mild Gaussian noise  (scanner grain)
    noise = rng.normal(0, rng.uniform(1, 6), img.shape).astype(np.float32)
    img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    # 3. Random crop + pad  (simulate slight scan misalignment, ±2% of each dim)
    h, w = img.shape[:2]
    max_dy = max(1, int(h * 0.02))
    max_dx = max(1, int(w * 0.02))
    top    = rng.integers(0, max_dy)
    left   = rng.integers(0, max_dx)
    bottom = rng.integers(0, max_dy)
    right  = rng.integers(0, max_dx)
    cropped = img[top:h-bottom, left:w-right]
    img = cv2.resize(cropped, (w, h), interpolation=cv2.INTER_LINEAR)

    # 4. JPEG compression artifacts  (typical for scanned docs saved as JPEG)
    quality = int(rng.integers(70, 95))
    _, enc = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    img = cv2.imdecode(enc, cv2.IMREAD_COLOR)

    return img

print('Augmentation pipeline ready.')

In [ ]:
source_images = [p for p in RAW_DIR.iterdir() if p.suffix.lower() in EXTENSIONS]
print(f'Source images: {len(source_images)}')
print(f'Expected dataset size: {len(source_images) * 4} (4 orientations × {len(source_images)} sources)')

In [ ]:
records = []

for img_path in tqdm(source_images, desc='Building dataset'):
    for degrees in [0, 90, 180, 270]:
        rotated = rotate_image(img_path, degrees)
        aug     = augment(rotated)

        out_name = f'{img_path.stem}_rot{degrees}.png'
        out_path = OUT_DIR / out_name
        cv2.imwrite(str(out_path), aug)

        records.append({
            'image_path': str(out_path.resolve()),
            'label':      DEGREES_TO_LABEL[degrees],
            'degrees':    degrees,
            'source':     img_path.name,
        })

df = pd.DataFrame(records)
csv_path = Path('../data/dataset.csv')
df.to_csv(csv_path, index=False)
print(f'Dataset saved → {csv_path}  ({len(df)} rows)')

In [ ]:
# ── Class balance verification ────────────────────────────────────────────────
counts = df.groupby(['degrees', 'label']).size().reset_index(name='count')
print(counts.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=counts, x='degrees', y='count', ax=ax,
            palette=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])
ax.set_title('Class Distribution (should be equal)')
ax.set_xlabel('Orientation (degrees)')
ax.set_ylabel('Image count')
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom')
plt.tight_layout()
plt.savefig(Path('../results/class_distribution.png'), dpi=150)
plt.show()

assert counts['count'].nunique() == 1, 'Classes are NOT balanced — investigate!'
print('Classes are balanced.')

In [ ]:
# ── Visual sanity check: one source × 4 orientations + augmentation ──────────
sample_src = source_images[0]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for col, deg in enumerate([0, 90, 180, 270]):
    raw = rotate_image(sample_src, deg)
    aug = augment(raw)

    axes[0, col].imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB))
    axes[0, col].set_title(f'{deg}° — original')
    axes[0, col].axis('off')

    axes[1, col].imshow(cv2.cvtColor(aug, cv2.COLOR_BGR2RGB))
    axes[1, col].set_title(f'{deg}° — augmented')
    axes[1, col].axis('off')

plt.suptitle(f'Source: {sample_src.name}', fontsize=12)
plt.tight_layout()
plt.savefig(Path('../results/augmentation_preview.png'), dpi=150)
plt.show()

In [ ]:
# ── Dataset summary ───────────────────────────────────────────────────────────
print('=== Dataset Summary ===')
print(f'Total images   : {len(df)}')
print(f'Per class      : {len(df) // 4}')
print(f'Unique sources : {df["source"].nunique()}')
print(f'CSV            : {csv_path}')
print(f'Images dir     : {OUT_DIR}')